# Ajuste fino do BERTimbau: classificador de intenção do Deliverius

Este notebook só chama comandos do repositório público, num commit fixo: o treino é
`research/intent/finetune.py`, o mesmo código que roda localmente e que os testes cobrem. Nenhuma
lógica de treino fica nas células.

- **O que roda aqui:** a busca da taxa de aprendizado (uma semente) e, se a cota de GPU der, três
  sementes na taxa escolhida. Cada execução guarda a época de melhor F1 macro na validação.
- **Dados:** o dataset sintético `G0m7s/deliverius-ai`, baixado na revisão fixada em
  `research/dataset.yaml`, com o SHA-256 de cada arquivo conferido. Só treino e validação são
  lidos; o teste escrito por pessoas nunca vem para cá.
- **Saída:** `/kaggle/working/runs/<taxa>-<semente>/` com os pesos em safetensors, o
  `tokenizer.json`, o `epochs.json` e o `result.json`. A exportação para ONNX, a calibração e a
  avaliação oficial rodam fora daqui, em CPU (`research/intent/export.py`).

**Acelerador:** GPU T4. Ative a internet do notebook (o `uv` baixa o Python 3.13 e as dependências).

In [ ]:
# O commit do repositório que este notebook reproduz. Trocar só ao refazer o treino.
REPOSITORY = "https://github.com/lucasg0m7s/deliverius-ai.git"
COMMIT = "PREENCHER_COM_O_COMMIT_PUBLICADO"
if len(COMMIT) != 40 or any(c not in "0123456789abcdef" for c in COMMIT):
    raise ValueError(COMMIT)  # not a full 40-character commit hash

In [ ]:
!pip install --quiet uv==0.13.6
!git clone --quiet {REPOSITORY} deliverius-ai
!git -C deliverius-ai checkout --quiet {COMMIT}
%cd deliverius-ai

In [ ]:
# Python 3.13 e as dependências exatas do uv.lock: a aplicação, o grupo research e o grupo finetune.
!uv sync --locked --group research --group finetune
!uv run python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0))"

## 1. Busca da taxa de aprendizado

Uma semente, cada taxa da grade de `research/intent/config.yaml`. O comando imprime a taxa de
melhor F1 macro na validação. Antes do treino ele mede quantas mensagens da validação passam de
128 tokens e escolhe a sequência máxima pela regra do design (128, ou 256 se passar de 5%).

In [ ]:
!uv run python -m research.intent.finetune search --out /kaggle/working/runs

## 2. Sementes na taxa escolhida

Copie a taxa que a célula anterior imprimiu. Se a cota de GPU acabar, uma semente basta para a
comparação; o relatório diz quantas rodaram.

In [ ]:
LEARNING_RATE = "PREENCHER_COM_A_TAXA_ESCOLHIDA"
RUNS = "/kaggle/working/runs"
!uv run python -m research.intent.finetune seeds --learning-rate {LEARNING_RATE} --out {RUNS}

In [ ]:
!ls -R /kaggle/working/runs | head -50
!cat /kaggle/working/runs/*/result.json